# Setting up Tensor Engine

In [24]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "TensorEngine" / "src" / "tensor_engine").is_dir() and (candidate / "FieldEquationsSolver" / "src" / "field_equations_solver").is_dir()), None)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Abre el notebook dentro del repositorio ModifiedTheoriesOfGravity_Research.")
WORKFLOW_ROOT = REPOSITORY_ROOT / "ResearchWorkflow"
OUTPUT_ROOT = WORKFLOW_ROOT / "outputs"
sys.path.insert(0, str(REPOSITORY_ROOT / "TensorEngine" / "src"))
sys.path.insert(0, str(REPOSITORY_ROOT / "FieldEquationsSolver" / "src"))
# Recarga el código local aunque este kernel haya importado una versión anterior.
for module_name in tuple(sys.modules):
    if module_name in {"tensor_engine", "field_equations_solver"} or module_name.startswith(("tensor_engine.", "field_equations_solver.")):
        del sys.modules[module_name]

from tensor_engine import (
    AnsatzSpecialization, DimensionSpec, DisplayPolicy, Function,
    LagrangianSourceSpec, Number, ParameterSpec, Scalar,
    TensorEngine, WolframXActBridge, draft4_angular_scalar_profile,
    draft4_circular_ansatz,
)

ansatz = draft4_circular_ansatz()
dimension = DimensionSpec(3)
VALIDAR_XACT = False  # Cambia a True si quieres validar cada caso con xAct.
display_policy = DisplayPolicy(
    factor=True, collect=True, together=True, canonicalize_indices=True,
    aggressive=False, enabled=True, max_nodes=4000,
)

def ejecutar(nombre, lagrangiano, *, ansatz_usado=None, parametros_extra=()):
    parametros = tuple(ParameterSpec(item) for item in parametros_extra)
    if "alpha" in lagrangiano:
        parametros = (ParameterSpec("alpha"), *parametros)
    model = LagrangianSourceSpec(
        name=nombre, expression=lagrangiano, dimension=dimension,
        parameters=parametros,
    ).compile()
    run = TensorEngine().run(
        model, ansatz=ansatz if ansatz_usado is None else ansatz_usado,
        output_root=OUTPUT_ROOT / "notebook_cases",
        display_policy=display_policy,
        wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None,
    )
    print(nombre, "|", run.status.value, "|", run.package.verification.summary)
    if run.export_bundle is not None:
        print("Bundle:", run.export_bundle.output_directory)
        if run.export_bundle.pdf_diagnostic:
            print("PDF:", run.export_bundle.pdf_diagnostic)
    return run

# Solve Field Equations

In [25]:
def solveFieldEq(condition, model):
    if condition:
        from field_equations_solver import solveFieldEquations
        from tensor_engine import AnsatzSpecialization, Scalar
        run_a_resolver = model
        solucion = solveFieldEquations(
            run_a_resolver,
            specialization=AnsatzSpecialization(scalar_field=Scalar("q")*Scalar("varphi")),
            output_root=OUTPUT_ROOT / "field_equations",
            display_policy=display_policy,
        )
        print(solucion.status, solucion.classification)
        print(solucion.output_directory)
# Omite specialization para Phi(r,varphi); usa Function("Phi", (Scalar("r"),)) para Phi(r).
# solve=False solo reduce; use_specialized=True usa la especialización previa de la corrida.

# 1. $R$

In [26]:
run_1 = ejecutar("R", "R")

R | partial | {'passed': 46, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-aa43c3ed782e


In [27]:
solveFieldEq(True, run_1)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-r*Derivative(f(r), (r, 2)) + Derivative(f(r), r)', 'Derivative(f(r), r)'], 'branch_note': 'Factores candidatos; deb

# 2. $R + \alpha R^2$

In [28]:
# 2. R + alpha R^2
run_2 = ejecutar("R_plus_alpha_times_R_squared", "R + alpha*R**2")

R_plus_alpha_times_R_squared | partial | {'passed': 46, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-plus-alpha-times-r-squared-7ba0c3689bd7


In [29]:
solveFieldEq(True, run_2)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 4, 'orders': {'f(r)': 4}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-4*alpha*r**3*f(r)*Derivative(f(r), (r, 4)) - 2*alpha*r**3*Derivative(f(r), r)*Derivative(f(r), (r

# 3. $R + \alpha R_{ab} R^{ab}$

In [30]:
run_3 = ejecutar("R_plus_alpha_times_RicciSq", "R + alpha*RicciSq")

R_plus_alpha_times_RicciSq | partial | {'passed': 45, 'failed': 0, 'undetermined': 3}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-plus-alpha-times-riccisq-553ab3598488


In [31]:
solveFieldEq(True, run_3)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 4, 'orders': {'f(r)': 4}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-4*alpha*r**3*f(r)*Derivative(f(r), (r, 4)) - 2*alpha*r**3*Derivative(f(r), r)*Derivative(f(r), (r

# 4. $R + \alpha R_{abcd} R^{abcd}$

In [32]:
run_4 = ejecutar("R_plus_alpha_times_RiemannSq", "R + alpha*RiemannSq")

R_plus_alpha_times_RiemannSq | partial | {'passed': 45, 'failed': 0, 'undetermined': 3}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-plus-alpha-times-riemannsq-110249a3cf81


In [33]:
solveFieldEq(True, run_4)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 4, 'orders': {'f(r)': 4}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-4*alpha*r**3*f(r)*Derivative(f(r), (r, 4)) - 2*alpha*r**3*Derivative(f(r), r)*Derivative(f(r), (r

# 5. $R + \alpha R_{ab} \nabla^a(\phi) \nabla^b(\phi)$

In [34]:
run_5 = ejecutar("R_plus_alpha_times_RicciUU", "R + alpha*RicciUU")

R_plus_alpha_times_RicciUU | partial | {'passed': 43, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-plus-alpha-times-ricciuu-a3c369602abd


In [35]:
solveFieldEq(True, run_5)

verified_with_pending_branches {'kind': 'DAE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': [], 'algebraic_constraints': [{'type': 'mul', 'factors': [{'type': 'number', 'numerator': 2, 'denominator': 1}, {'type': 'scalar', 'name': 'alpha'}, {'type': 'power', 'base': {'type': 'scalar', 'name': 'q'}, 'exponent': {'type': 'number', 'numerator': 2, 'denominator': 1}}, {'type': 'power', 'base': {'type': 'scalar', 'name': 'r'}, 'exponent': {'type': 'number', 'numerator': -4, 'denominator': 1}}, {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}]}], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha', 'q'], 'projection_source': 'Componentes genéricas ausentes: proyectadas por primera vez desde E_ab/E_phi almacenado

# 6. $R + \alpha R X$

In [36]:
run_6 = ejecutar("R_plus_alpha_times_RX", "R + alpha*R*X")

R_plus_alpha_times_RX | partial | {'passed': 46, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\r-plus-alpha-times-rx-c414b3705791


In [37]:
solveFieldEq(True, run_6)

verified_with_pending_branches {'kind': 'DAE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': [], 'algebraic_constraints': [{'type': 'mul', 'factors': [{'type': 'number', 'numerator': 6, 'denominator': 1}, {'type': 'scalar', 'name': 'alpha'}, {'type': 'power', 'base': {'type': 'scalar', 'name': 'q'}, 'exponent': {'type': 'number', 'numerator': 2, 'denominator': 1}}, {'type': 'power', 'base': {'type': 'scalar', 'name': 'r'}, 'exponent': {'type': 'number', 'numerator': -4, 'denominator': 1}}, {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}]}], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'typ

# 7. $R + \alpha R_{abcd} \nabla^a(\phi) \nabla^c(\phi) \nabla^b(\phi) \nabla^d(\phi)$

In [38]:
# El término cuártico puede anularse por las antisimetrías de R_abcd.
quartic_riemann_gradient = 'contract(Riemann("a","b","c","d"), metric("a","e"), gradient("e"), metric("c","f"), gradient("f"), metric("b","g"), gradient("g"), metric("d","h"), gradient("h"))'
run_7 = ejecutar("case_07_R_plus_RiemannGrad4", f"R + alpha*{quartic_riemann_gradient}")

case_07_R_plus_RiemannGrad4 | partial | {'passed': 46, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\ResearchWorkflow\outputs\notebook_cases\case-07-r-plus-riemanngrad4-4da30f66c796


In [39]:
solveFieldEq(True, run_7)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-r*Derivative(f(r), (r, 2)) + Derivative(f(r), r)', 'Derivative(f(r), r)'], 'branch_note': 'Factores candidatos; deb

# Especialización opcional posterior: $\phi = p\varphi$.

In [40]:
'''
ansatz_p_varphi = ansatz.specialize_scalar(
    draft4_angular_scalar_profile("p"),
    assumptions=("phi=p*varphi",),
)
run_p_varphi = ejecutar(
    "RicciUU_with_explicit_p_varphi_profile", "R + alpha*RicciUU",
    ansatz_usado=ansatz_p_varphi, parametros_extra=("p",),
)
'''

'\nansatz_p_varphi = ansatz.specialize_scalar(\n    draft4_angular_scalar_profile("p"),\n    assumptions=("phi=p*varphi",),\n)\nrun_p_varphi = ejecutar(\n    "RicciUU_with_explicit_p_varphi_profile", "R + alpha*RicciUU",\n    ansatz_usado=ansatz_p_varphi, parametros_extra=("p",),\n)\n'

# Draft 4 - Caso 0

In [41]:
# Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_0 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_0.chart.coordinates
ell, mass = Scalar("ell"), Scalar("lambda")
f_input = r**2 / ell**2 - mass
phi_input = Number(0)  # El escalar está desacoplado en este caso.
source = LagrangianSourceSpec(name="draft4_case_0", expression="R + 2/ell**2", dimension=DimensionSpec(3), parameters=(ParameterSpec("ell"), ParameterSpec("lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_0_specialized")
run_case_0 = TensorEngine().run(source.compile(), ansatz=ansatz_case_0, specialization=specialization, output_root=OUTPUT_ROOT / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_0.export_bundle.output_directory if run_case_0.export_bundle else run_case_0.status.value)

C:\Investigacion\ResearchWorkflow\outputs\draft4_cases\draft4-case-0-94744d3948d8


In [42]:
solveFieldEq(True, run_case_0)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'ell', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': ['q'], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['ell', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-r*Derivative(f(r), (r, 2)) + Derivative(f(r), r)', 'ell**2*Derivative(f(r), r) - 2*r'], 'branch_note'

# Draft 4 - Caso 1

In [43]:
# Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_1 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_1.chart.coordinates
ell, alpha1, p, r0, mass = (Scalar(name) for name in ("ell", "alpha1", "p", "r0", "lambda"))
f_input = r**2 / ell**2 - mass - alpha1*p**2*Function("log", (r/r0,))
phi_input = p*varphi
source = LagrangianSourceSpec(name="draft4_case_1", expression="R + 2/ell**2 - alpha1*X", dimension=DimensionSpec(3), parameters=tuple(ParameterSpec(name) for name in ("ell", "alpha1", "p", "r0", "lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_1_specialized")
run_case_1 = TensorEngine().run(source.compile(), ansatz=ansatz_case_1, specialization=specialization, output_root=OUTPUT_ROOT / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_1.export_bundle.output_directory if run_case_1.export_bundle else run_case_1.status.value)

C:\Investigacion\ResearchWorkflow\outputs\draft4_cases\draft4-case-1-f9726e0182f4


In [44]:
solveFieldEq(True, run_case_1)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'alpha1', 'ell', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': [], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['alpha1', 'ell', 'q'], 'projection_source': 'Componentes existentes reutilizadas.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['2*alpha1*q**2 - r**2*Derivative(f(r), (r, 2)) + r*Derivative(f(r), r)', 'alpha1*ell**

# Draft 4 - Caso 2

In [45]:
# Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_2 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_2.chart.coordinates
ell, beta0, p, mass = (Scalar(name) for name in ("ell", "beta0", "p", "lambda"))
f_input = (r**2 / ell**2 - mass) / (1 + beta0*p**2*ell**2/r**2)
phi_input = p*varphi
source = LagrangianSourceSpec(name="draft4_case_2", expression="R + 2/ell**2 + ell**2*beta0*(3*RicciUU - X*R)", dimension=DimensionSpec(3), parameters=tuple(ParameterSpec(name) for name in ("ell", "beta0", "p", "lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_2_specialized")
run_case_2 = TensorEngine().run(source.compile(), ansatz=ansatz_case_2, specialization=specialization, output_root=OUTPUT_ROOT / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_2.export_bundle.output_directory if run_case_2.export_bundle else run_case_2.status.value)

C:\Investigacion\ResearchWorkflow\outputs\draft4_cases\draft4-case-2-4c98cf607ace


In [46]:
solveFieldEq(True, run_case_2)

verified_with_pending_branches {'kind': 'ODE', 'contains_pde': False, 'unknowns': ['f(r)', 'beta0', 'ell', 'q'], 'independent_variables': ['r'], 'max_derivative_order': 2, 'orders': {'f(r)': 2}, 'unconstrained_unknowns': [], 'algebraic_constraints': [], 'parameter_constraints': [], 'independence_scope': 'Solo dependencia lineal exacta sobre Q; independencia diferencial no certificada.', 'functions': ['f(r)'], 'parameters': ['beta0', 'ell', 'q'], 'projection_source': 'Componentes genéricas ausentes: proyectadas por primera vez desde E_ab/E_phi almacenados, con el perfil solicitado.', 'domain_assumptions': [{'op': 'gt', 'lhs': {'type': 'scalar', 'name': 'r'}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'r>0'}, {'op': 'ne', 'lhs': {'type': 'function', 'name': 'f', 'arguments': [{'type': 'scalar', 'name': 'r'}]}, 'rhs': {'type': 'number', 'numerator': 0, 'denominator': 1}, 'source': 'f(r)!=0'}], 'uninterpreted_assumptions': [], 'possible_zero_factors': ['-beta0*e